<a href="https://colab.research.google.com/github/josephineabioye/msc-skincare-reaction-prediction/blob/main/notebooks/01_product_data_loading_and_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Notebook Setup Cell
from google.colab import userdata, drive
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = 'josephineabioye'
REPO_NAME = 'msc-skincare-reaction-prediction'

if not os.path.exists(REPO_NAME):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
    %cd {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull

!git config user.email "josephineabioye@yahoo.com"
!git config user.name "Josephine Abioye"

drive.mount('/content/drive', force_remount=False)

DRIVE_PROJECT = '/content/drive/MyDrive/msc-skincare-project'
DATA_RAW = f'{DRIVE_PROJECT}/data/raw'
DATA_PROCESSED = f'{DRIVE_PROJECT}/data/processed'

print(f"Working in: {os.getcwd()}")

Cloning into 'msc-skincare-reaction-prediction'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 56 (delta 22), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 21.36 KiB | 7.12 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/msc-skincare-reaction-prediction
Mounted at /content/drive
Working in: /content/msc-skincare-reaction-prediction


In [2]:
import os
for f in sorted(os.listdir(DATA_RAW)):
    print(f)

product_info.csv
reviews_0-250.csv
reviews_1250-end.csv
reviews_250-500.csv
reviews_500-750.csv
reviews_750-1250.csv


In [4]:
import pandas as pd

products = pd.read_csv(f"{DATA_RAW}/product_info.csv")

print("Product cvs shape (rows, columns):", products.shape)
print("\nColumn names:")
print(products.columns.tolist())

Product cvs shape (rows, columns): (8494, 27)

Column names:
['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']


In [6]:
# Looking up the ingredients column for the first few products

for i in range(5):
    print(f"Product {i}: {products['ingredients'].iloc[i]} ")
    print()

Product 0: ['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, Benzyl Salicylate, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Benzl Benzoate, Citral, Geraniol, Eugenol, Benzyl Alcohol, Farnesol, Citronellol, Isoeugenol.', 'Invisible Post Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Salicylate, Butyl Methoxydibenzoylmethane, Benzyl Benzoate, Citral, Coumarin, Limonene, Alpha-Isomethyl Ionone, Linalool.', 'Kashbah Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Coumarin, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, D-Limonene, Eugenol, Linalool, Citronellol, Geraniol, Cinnamal, Citral.', 'L’Air Barbes Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Benzyl Salicylate, D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylat

In [10]:
## Noticed the ingredient structure for product 0 is a bit different,
## I want to check if it's linked to the product type.

products[['product_name', 'primary_category', 'secondary_category', 'ingredients']].head(5)

,product_name,primary_category,secondary_category,ingredients
0,Fragrance Discovery Set,Fragrance,Value & Gift Sets,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A..."
1,La Habana Eau de Parfum,Fragrance,Women,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra..."
2,Rainbow Bar Eau de Parfum,Fragrance,Women,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra..."
3,Kasbah Eau de Parfum,Fragrance,Women,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra..."
4,Purple Haze Eau de Parfum,Fragrance,Women,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra..."


In [11]:
# Product 0 is different because its a set and has more than one products
# Now let's see what primary categories exist and how many products each has,
# so we know the exact label to filter skincare on

products['primary_category'].value_counts()

,count
primary_category,
Skincare,2420
Makeup,2369
Hair,1464
Fragrance,1432
Bath & Body,405
Mini Size,288
Men,60
Tools & Brushes,52
Gifts,4


In [12]:
## Filtering down to only skincare products as that is the focus

skincare = products[products['primary_category'] == 'Skincare'].copy()
print("Skincare Products:", skincare.shape[0])
print("Columns retained:", skincare.shape[1])

Skincare Products: 2420
Columns retained: 27


In [15]:
# Check if all entry has ingredients listed and
# Check if their are skincare sets similar to product 0 from earlier(multivariant)

missing_ingredients = skincare['ingredients'].isna().sum()
print("Skincare Products with no ingredients listed:", missing_ingredients)

has_ingredients = skincare[skincare['ingredients'].notna()]

multivariant = has_ingredients['ingredients'].str.contains(":'", regex=False).sum()

print("Skincare products with ingredients:", has_ingredients.shape[0])
print("How many multi-variant (gift-set style)?:", multivariant)
print("remove the multivariants (the rest):", has_ingredients.shape[0] - multivariant)


Skincare Products with no ingredients listed: 134
Skincare products with ingredients: 2286
How many multi-variant (gift-set style)?: 217
remove the multivariants (the rest): 2069


In [19]:
# Rough spot-check of the multivariant set
# Print the first 15

multivariant_sets = has_ingredients[has_ingredients['ingredients'].str.contains(":'", regex=False)]

multivariant_sets[['product_name', 'secondary_category', 'ingredients']].head(15)

,product_name,secondary_category,ingredients
97,10 Day Results Kit,Value & Gift Sets,"['GENIUS Liquid Collagen:', 'Collagen (Vegan),..."
143,Peel and Plump Skin-Smoothing Duo,Value & Gift Sets,['Melt Moisturizer with Bakuchiol and Squalane...
431,The Skin Renewal System,Value & Gift Sets,['The Cream Cleansing Gel with TFC8 Gentle Cle...
459,Get That Glow - GloPRO Facial Microneedling Di...,Value & Gift Sets,"['The Balance pH Balancing Gel Cleanser:', 'Wa..."
465,The Eyelighter Concentrate,Eye Care,"['Water/Aqua/Eau, Dimethicone, Propanediol, Vi..."
467,GLOfacial Hydro-Infusion Pore Cleansing + Blue...,Value & Gift Sets,['GLOfacial Salicylic Acid & Hyaluronic Acid S...
474,Rejuvenating Scalp + Fuller Hair Therapy Set,Value & Gift Sets,"['Healthy Scalp Serum:', 'Water/Aqua/Eau, Glyc..."
475,R45 The Lift 3-Phase Advanced Neck Contouring ...,Treatments,"['Phase One:', 'Water, Ethylhexyl Stearate, Gl..."
485,Head-To-Toe AfterGLO Set,Value & Gift Sets,"['The Zenbubble Gel Cream:', 'Water/Aqua/Eau, ..."
487,Eye Want It All Face + Eye Microneedling Set,Value & Gift Sets,['The Quench Eye Reviving Quadralipid Eye Balm...
